In [1]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261102 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# COMMAND 

In [2]:
%%writefile gds2agds.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

args <- commandArgs(trailingOnly = TRUE)
# Retrieves command-line arguments passed to the R script.
# trailingOnly = TRUE means we only receive arguments supplied by the user,
# rather than R's own command-line options.

# Expected arguments:
#   args[1] = input GDS file
#   args[2] = functional annotation CSV
#   args[3] = output GDS file
#   args[4] = chromosome number

gds_in <- args[1]
anno_file <- args[2]
gds_out <- args[3]
chr <- as.integer(args[4])

# Print the inputs so that the log clearly shows exactly what was used.
cat("CHROMOSOME:", chr, "\n")
cat("GENOTYPE INPUT:", gds_in, "\n")
cat("ANNOTATION INPUT:", anno_file, "\n")
cat("OUTPUT:", gds_out, "\n")

##########################################################################
# LOAD REQUIRED PACKAGES
##########################################################################

# Load packages. 
library(gdsfmt)
library(SeqArray)
library(SeqVarTools)
library(readr)

##########################################################################
# COPY INPUT GDS TO OUTPUT
##########################################################################

cat("Copying GDS to output...\n")

# Makes a copy of the original GDS.
file.copy(
    from = gds_in,
    to = gds_out,
    overwrite = TRUE
)

cat("GDS copied successfully.\n")

##########################################################################
# READ ANNOTATION
##########################################################################

cat("Reading functional annotation...\n")
# Reads the annotation CSV into an R data frame/tibble.
# The col_types argument explicitly tells read_csv() what type each
# column should be.

# For example:
#   col_character() = text/string
#   col_double()     = numeric/double
# There are 22 column definitions here, so the annotation file is
# expected to contain 22 columns.

FunctionalAnnotation <- read_csv(
    anno_file,
    col_types = list(
        col_character(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_double(),
        col_character(),
        col_character(),
        col_character(),
        col_double(),
        col_character(),
        col_character(),
        col_character(),
        col_character(),
        col_character(),
        col_double(),
        col_double(),
        col_character()
    )
)

# Prints the number of rows × columns in the annotation table.
cat(
    "Annotation dimensions:",
    dim(FunctionalAnnotation)[1],
    "x",
    dim(FunctionalAnnotation)[2],
    "\n"
)

##########################################################################
# RENAME COLUMNS
##########################################################################

# Renames column 2 to "apc_conservation".
# Renames column 7 to "apc_local_nucleotide_diversity".
# Renames column 9 to "apc_protein_function".
colnames(FunctionalAnnotation)[2] <- "apc_conservation"
colnames(FunctionalAnnotation)[7] <- "apc_local_nucleotide_diversity"
colnames(FunctionalAnnotation)[9] <- "apc_protein_function"

##########################################################################
# OPEN OUTPUT GDS
##########################################################################

cat("Opening output GDS...\n")

# Opens the copied GDS for modification.
# readonly = FALSE is critical:
# it allows the script to add the FunctionalAnnotation node.
genofile <- seqOpen(
    gds_out,
    readonly = FALSE
)

# Prints information about the opened SeqArray GDS.
print(genofile)

##########################################################################
# ADD FUNCTIONAL ANNOTATION
##########################################################################

cat("Adding FunctionalAnnotation...\n")

# Locates the existing "annotation/info" node inside the GDS.
Anno.folder <- index.gdsn(
    genofile,
    "annotation/info"
)

# Adds the entire FunctionalAnnotation data frame to:
#   annotation/info/FunctionalAnnotation
# inside the GDS.
# "FunctionalAnnotation" becomes the name of the new GDS node.
# val = FunctionalAnnotation
#     The actual annotation table being stored.
# compress = "LZMA_ra"
#     Compresses the stored annotation data using LZMA.
# closezip = TRUE
#     Closes/finalizes the compressed storage after writing.

add.gdsn(
    Anno.folder,
    "FunctionalAnnotation",
    val = FunctionalAnnotation,
    compress = "LZMA_ra",
    closezip = TRUE
)

##########################################################################
# CLOSE GDS
##########################################################################

# Prints the GDS object again before closing.
print(genofile) 

# Closes the GDS file and releases the file handle.
seqClose(genofile)

# Final status messages indicating that the script completed and
# showing where the resulting GDS was written.
cat("Finished successfully.\n")
cat("OUTPUT:", gds_out, "\n")

Overwriting gds2agds.R


In [3]:
# Save script to workspace bucket 
!gcloud storage cp gds2agds.R ${WORKSPACE_BUCKET}/data/scripts/

Copying file://gds2agds.R to gs://rw-migration-aou-rw-9f879fc4/data/scripts/gds2agds.R
  Completed files 1/1 | 5.0kiB/5.0kiB                                          


In [4]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/gds2agds.R 

#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

args <- commandArgs(trailingOnly = TRUE)
# Retrieves command-line arguments passed to the R script.
# trailingOnly = TRUE means we only receive arguments supplied by the user,
# rather than R's own command-line options.

# Expected arguments:
#   args[1] = input GDS file
#   args[2] = functional annotation CSV
#   args[3] = output GDS file
#   args[4] = chromosome number

gds_in <- args[1]
anno_file <- args[2]
gds_out <- args[3]
chr <- as.integer(args[4])

# Print the inputs so that the log clearly shows exactly what was used.
cat("CHROMOSOME:", chr, "\n")
cat("GENOTYPE INPUT:", gds_in, "\n")
cat("ANNOTATION INPUT:", anno_file, "\n")
cat("OUTPUT:", gds_out, "\n")

##########################################################################
# LOAD REQUIRED PACKAGES
############################################

# SUBMIT COMMAND 

In [5]:
import subprocess
import pandas as pd

# Create dataframe of size of exome plink files in GB 
chromosome = []
sizes = [] 

# Loop for all chromosome 
for chrom in [17]:
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{chrom}.gds",
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


   chromosome      size
0          17  0.775508


In [7]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "gds2agds.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

MAX_DISK_SIZE = 10 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 3 # Set additional disk size for each chromosome 

# Loop to submit dsub command
for CHROM in [17]: 
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (
        round(
            float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"
                ].iloc[0]
            )
        )
        + EXTRA_DISK_SIZE
    )

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # File paths
    GDS = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{CHROM}.gds"
    ANNOTATION = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/annotations/chr{CHROM}/anno_chr{CHROM}_STAARpipeline.csv"
    OUTPUT = f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/annotated_chr{CHROM}.gds"

    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$GDS" "$ANNOTATION" "$OUTPUT" "$CHROM"'
    print(R_COMMAND)
    command = [
        "aou_dsub",
            "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/vieno99/favor:v1.0",
            "--boot-disk-size", "5",
            "--disk-size", str(DISK_SIZE),
            "--machine-type", "n1-standard-1",
            "--logging", LOGGING,
            "--input", f"SCRIPT={PATH}",
            "--input", f"GDS={GDS}",
            "--input", f"ANNOTATION={ANNOTATION}",
            "--output", f"OUTPUT={OUTPUT}",
            "--env", f"CHROM={CHROM}",
            "--command", R_COMMAND,
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 17
Requesting disk size: 4
Rscript "$SCRIPT" "$GDS" "$ANNOTATION" "$OUTPUT" "$CHROM"
rscript--vienose--261104-011306-11

Job properties:
  job-id: rscript--vienose--261104-011306-11
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/rscript--vienose--261104-011306-11-0-0
Launched job-id: rscript--vienose--261104-011306-11
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261104-011306-11' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261104-011306-11' --users 'vienose'



In [26]:
! dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --users 'vienose' --status '*' | head 

Job Name         Status    Last Update
---------------  --------  --------------
rscript          SUCCESS   11-04 01:20:55
filtered-hig...  SUCCESS   11-03 22:31:52
filtered-hig...  SUCCESS   11-03 22:31:38
filtered-hig...  SUCCESS   11-03 22:32:10
filtered-hig...  SUCCESS   11-03 22:32:09
filtered-hig...  SUCCESS   11-03 22:19:46
filtered-hig...  CANCELED  11-03 22:14:02
saige-gene-s...  SUCCESS   11-03 21:42:00


In [27]:
!gsutil ls gs://rw-migration-aou-rw-9f879fc4/data/logging | grep 'rscript--vienose--261104-011306-11'

gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261104-011306-11-stderr.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261104-011306-11-stdout.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261104-011306-11.log


In [28]:
!gcloud storage cat gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261104-011306-11.log

[batch_task_logs]2026/11/04 01:16:19 INFO: [task_id:task/rscript--vienose--f1dbb32e-84e5-4a2200-group0-0/0/0,runnable_index:1] 2026-11-04 01:16:19 INFO: mkdir -m 777 -p /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/scripts
[batch_task_logs]2026/11/04 01:16:19 INFO: [task_id:task/rscript--vienose--f1dbb32e-84e5-4a2200-group0-0/0/0,runnable_index:1] 2026-11-04 01:16:19 INFO: mkdir -m 777 -p /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/staar/annotations/chr17
[batch_task_logs]2026/11/04 01:16:19 INFO: [task_id:task/rscript--vienose--f1dbb32e-84e5-4a2200-group0-0/0/0,runnable_index:1] 2026-11-04 01:16:19 INFO: mkdir -m 777 -p /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/staar/gds
[batch_task_logs]2026/11/04 01:16:19 INFO: [task_id:task/rscript--vienose--f1dbb32e-84e5-4a2200-group0-0/0/0,runnable_index:1] 2026-11-04 01:16:19 INFO: mkdir -m 777 -p /mnt/data/output/gs/rw-migration-aou-rw-9f879fc4/data/staar/gds
[batch_task_logs]2026/11/04 01:16:20 INFO: [task_id:task/rs